# BM25 for Resume–Job Matching

1) Loading the datasets.

In [ ]:
from datasets import load_dataset

# Job descriptions dataset
ds = load_dataset("lang-uk/recruitment-dataset-job-descriptions-english")
df_jobs = ds["train"].to_pandas()
df_jobs.to_csv("job-descriptions.csv", index=False)

# For the resume dataset
# Place CareerCorpus.xlsx in the same working directory before running the notebook

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/146M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/141897 [00:00<?, ? examples/s]

2) Reading the datasets and installing packages for BM25

In [ ]:
!pip install rank-bm25
from rank_bm25 import BM25Okapi
import numpy as np
import pandas as pd

df_resumes = pd.read_excel("CareerCorpus.xlsx")
df_jobs = pd.read_csv("job-descriptions.csv")

3) Inspecting the datasets

In [ ]:
print(df_resumes.head())
print(df_resumes.columns)

print(df_jobs.head())
print(df_jobs.columns)

4) Defining the text for each dataset and putting them together

Resume dataset relevant columns:
*   *Education*
*   *Skills and Achievements*
*   *Experience*
These together form the CV content.

Jobs dataset relevant columns:
*   *Position*
*   *Long Description*

Keeping the Domain and Job_type because they may help later with inspecting or evaluating results.

In [ ]:
# For the resume dataset
resumes = df_resumes.copy()

resumes["text"] = (
    resumes["Education"].fillna('') + " " +
    resumes["Skills and Achievements"].fillna('') + " " +
    resumes["Experience"].fillna('')
)

resumes = resumes[["text", "Domain", "Job_type"]]

# For the jobs dataset
jobs = df_jobs.copy()

jobs["text"] = (
    jobs["Position"].fillna('') + " " +
    jobs["Long Description"].fillna('')
)

jobs = jobs[["text", "Primary Keyword"]]

# Removing empty rows
resumes = resumes.dropna(subset=["text"]).copy()
jobs = jobs.dropna(subset=["text"]).copy()

5) Minimal cleaning

In [ ]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

resumes["cleaned"] = resumes["text"].apply(clean_text)
jobs["cleaned"] = jobs["text"].apply(clean_text)

6) Tokenizing the resumes and building the BM25 index

In [ ]:
resume_tokens = [text.split() for text in resumes["cleaned"]]

bm25 = BM25Okapi(resume_tokens)


**Testing** **1**

In [ ]:
job_index = 0

query = jobs.iloc[job_index]["cleaned"]
query_tokens = query.split()
scores = bm25.get_scores(query_tokens)

#Ranking the resumes
resumes["score"] = scores

#Sorting from highest to lowest
ranked = resumes.sort_values(by="score", ascending=False)

#Creating a shorter preview column
ranked["preview"] = ranked["text"].str[:300]

print("JOB:")
print(jobs.iloc[job_index]["text"][:500])

print("\nTOP MATCHES:")
print(ranked[["preview", "score"]].head(10))

JOB:
10 + Blockchain Nodes / Masternodes to set up *Requirements*

We're looking for a long term collaboration with someone that has an experience in crypto, masternodes, nodes, validators etc. We need to set up:

Kyber Network
Nebulas
SecretNetwork
Tron
Aion
DeFiChain
EOS
TomoChain
Elrond
IRISnet
IoTeX
Terra
ChainX
Thorchain

Succesful candidates will have an opportunity to get more jobs and long term collaboration.

TOP MATCHES:
                                               preview      score
155  Elementary Teaching Credential, University of ...  28.204383
183  Associate of Arts — Business, Jones Internatio...  27.333054
152  M.Ed., Teaching—Virginia Commonwealth Universi...  23.659949
190  M.A. (Guidance & Counseling), University of Te...  21.041329
188  Master’s in Education, Government College of E...  16.173876
198  B.S. Education, University of Central Missouri...  15.292390
175  M.S. Education — Liberty University (2017); B....  15.033335
286  B.Sc. in CSE, United Internation

**Testing** **2**

In [ ]:
job_index = 6

query = jobs.iloc[job_index]["cleaned"]
query_tokens = query.split()
scores = bm25.get_scores(query_tokens)

#Ranking the resumes
resumes["score"] = scores

#Sorting from highest to lowest
ranked = resumes.sort_values(by="score", ascending=False)

#Creating a shorter preview column
ranked["preview"] = ranked["text"].str[:300]

print("JOB:")
print(jobs.iloc[job_index]["text"][:500])

print("\nTOP MATCHES:")
print(ranked[["preview", "score"]].head(10))

JOB:
1806/03 Sales Manager **Requirements:**
•⠀Experience in IT sales 1+ year
•⠀Generic understanding of IT technologies and Software Development process
•⠀Knowledge of sales techniques, sales markets
•⠀Good communication and negotiation skills
•⠀Fluent written and good spoken English

**We offer:**
•⠀Friendly teams, experienced colleagues, and perfect work equipment
•⠀Opportunities for career growth and raising professional skills
•⠀Competitive compensation and regular results-based salary

TOP MATCHES:
                                               preview      score
183  Associate of Arts — Business, Jones Internatio...  60.782057
1    High School Diploma, Federal Way Senior High S...  51.795835
5    High School Diploma (Math), University of Cali...  49.963763
101  BBA in Accountancy, Western Michigan Universit...  49.057039
191  M.A. in Math Education, Southwestern Universit...  48.822075
188  Master’s in Education, Government College of E...  48.635648
218  BSBA in Marketing, Univ

**Testing** **3**

In [18]:
job_index = 82

query = jobs.iloc[job_index]["cleaned"]
query_tokens = query.split()
scores = bm25.get_scores(query_tokens)

#Ranking the resumes
resumes["score"] = scores

#Sorting from highest to lowest
ranked = resumes.sort_values(by="score", ascending=False)

#Creating a shorter preview column
ranked["preview"] = ranked["text"].str[:300]

print("JOB:")
print(jobs.iloc[job_index]["text"][:500])

print("\nTOP MATCHES:")
print(ranked[["preview", "score"]].head(10))

JOB:
2D-, 3D-Animator position Onlyplay is a fast-growing gaming product development company. Our team develops high-quality games to launch them on popular platforms.

We are inspired by new ideas and bring trends to the gaming world. For the coming years, we face serious goals and many interesting tasks.

With the aim of reaching new horizons in the gaming industry, we are announcing a contest for the position 2D-, 3D-Animator, who will join us in Kyiv on a full-time basis. We are a product com

TOP MATCHES:
                                               preview       score
183  Associate of Arts — Business, Jones Internatio...  146.904038
191  M.A. in Math Education, Southwestern Universit...  134.846828
188  Master’s in Education, Government College of E...  132.476620
190  M.A. (Guidance & Counseling), University of Te...  130.007409
237  A.A. Office Systems Technology, Horry Georgeto...  121.805020
287  B.Arch in Architecture, Chittagong Univ. of En...  119.437849
1    High Schoo

**Testing** **4**

In [17]:
job_index = 24

query = jobs.iloc[job_index]["cleaned"]
query_tokens = query.split()
scores = bm25.get_scores(query_tokens)

#Ranking the resumes
resumes["score"] = scores

#Sorting from highest to lowest
ranked = resumes.sort_values(by="score", ascending=False)

#Creating a shorter preview column
ranked["preview"] = ranked["text"].str[:300]

print("JOB:")
print(jobs.iloc[job_index]["text"][:500])

print("\nTOP MATCHES:")
print(ranked[["preview", "score"]].head(10))

JOB:
1.Middle/Senior Embedded C/C++ The CHI team is looking for a Middle/Senior Embedded C/C++

About the client:
The leading global supplier of embedded and connected software for the automotive industry.
Client`s company develops the award-winning iGO Navigation Engine, which is flexible, designed for automotive use, and assists in prototyping and developing proofs of concept, whether it’s navigation, connectivity, HMI, or the complete in-car experience.
For deep integration, he configures his t

TOP MATCHES:
                                               preview       score
1    High School Diploma, Federal Way Senior High S...  178.045167
191  M.A. in Math Education, Southwestern Universit...  161.641561
257  B.Sc. in Computer Science & Engineering (2020–...  156.168286
278  Ph.D. in Computer Science, Louisiana Tech Univ...  154.786816
237  A.A. Office Systems Technology, Horry Georgeto...  154.624678
187  Bachelor of Science in Reading, Delta State Un...  152.971856
188  Master’s 

**Testing** **5** (with custom job description)

In [ ]:
custom_job = """
Looking for a data analyst with Python, SQL, data visualization,
machine learning, and statistics experience.
"""
custom_clean = clean_text(custom_job)

#Tokenize
custom_tokens = custom_clean.split()
#Get
scores = bm25.get_scores(custom_tokens)

#Ranking
resumes["score"] = scores
ranked = resumes.sort_values(by="score", ascending=False)
ranked["preview"] = ranked["text"].str[:300]

print("CUSTOM JOB:")
print(custom_job)

print("\nTOP MATCHES:")
print(ranked[["preview", "score"]].head(10))

CUSTOM JOB:

Looking for a data analyst with Python, SQL, data visualization,
machine learning, and statistics experience.


TOP MATCHES:
                                               preview      score
287  B.Arch in Architecture, Chittagong Univ. of En...  15.536398
254  B.Sc. in CSE, East West University (2017–2022)...  14.511345
280  M.Sc. in Food Processing & Preservation (2025–...  13.969958
249  B.S. in Information Systems, Northeastern Univ...  13.154484
261  B.Sc. in Software Engineering, Shahjalal Unive...  12.620377
252  M.Sc. in CSE (2023–2026) and B.Sc. in CSE (201...  12.436049
298  B.Sc. in Electronics & Telecommunication Engin...  11.856943
295  B.Sc. in Computer Science & Engineering, East ...  10.878739
187  Bachelor of Science in Reading, Delta State Un...  10.592782
235  B.S. Metallurgy & Materials Engineering, Color...   9.864466


7) Saving the rankings for the tested dataset jobs and the custom job

In [19]:
k = 10
all_results = []

# for dataset jobs
tested_jobs = [0, 6, 82, 24]

for job_index in tested_jobs:
    query = jobs.iloc[job_index]["cleaned"]
    query_tokens = query.split()
    scores = bm25.get_scores(query_tokens)

    top_k_idx = np.argsort(scores)[-k:][::-1]

    for rank, idx in enumerate(top_k_idx, start=1):
        all_results.append({
            "job_id": f"dataset_{job_index}",
            "job_type": "dataset",
            "job_text": jobs.iloc[job_index]["text"][:300],
            "rank": rank,
            "resume_index": idx,
            "score": scores[idx],
            "resume_domain": resumes.iloc[idx]["Domain"],
            "resume_preview": resumes.iloc[idx]["text"][:300]
        })

# for the custom job desciption
custom_job = """
Looking for a data analyst with Python, SQL, data visualization,
machine learning, and statistics experience.
"""

custom_clean = clean_text(custom_job)
custom_tokens = custom_clean.split()
scores = bm25.get_scores(custom_tokens)

top_k_idx = np.argsort(scores)[-k:][::-1]

for rank, idx in enumerate(top_k_idx, start=1):
    all_results.append({
        "job_id": "custom_0",
        "job_type": "custom",
        "job_text": custom_job[:300],
        "rank": rank,
        "resume_index": idx,
        "score": scores[idx],
        "resume_domain": resumes.iloc[idx]["Domain"],
        "resume_preview": resumes.iloc[idx]["text"][:300]
    })

final_df = pd.DataFrame(all_results)
final_df.to_csv("BM25_Results.csv", index=False)

final_df.head(20)

,job_id,job_type,job_text,rank,resume_index,score,resume_domain,resume_preview
0,dataset_0,dataset,10 + Blockchain Nodes / Masternodes to set up ...,1,155,28.204383,TEACHER,"Elementary Teaching Credential, University of ..."
1,dataset_0,dataset,10 + Blockchain Nodes / Masternodes to set up ...,2,183,27.333054,TEACHER,"Associate of Arts — Business, Jones Internatio..."
2,dataset_0,dataset,10 + Blockchain Nodes / Masternodes to set up ...,3,152,23.659949,TEACHER,"M.Ed., Teaching—Virginia Commonwealth Universi..."
3,dataset_0,dataset,10 + Blockchain Nodes / Masternodes to set up ...,4,190,21.041329,TEACHER,"M.A. (Guidance & Counseling), University of Te..."
4,dataset_0,dataset,10 + Blockchain Nodes / Masternodes to set up ...,5,188,16.173876,TEACHER,"Master’s in Education, Government College of E..."
5,dataset_0,dataset,10 + Blockchain Nodes / Masternodes to set up ...,6,198,15.292390,TEACHER,"B.S. Education, University of Central Missouri..."
6,dataset_0,dataset,10 + Blockchain Nodes / Masternodes to set up ...,7,175,15.033335,TEACHER,M.S. Education — Liberty University (2017); B....
7,dataset_0,dataset,10 + Blockchain Nodes / Masternodes to set up ...,8,286,14.415465,Research Assistant,"B.Sc. in CSE, United International Univ. (CGPA..."
8,dataset_0,dataset,10 + Blockchain Nodes / Masternodes to set up ...,9,301,13.803801,Research Assistant,B.Sc. in Computer Science & Engineering (CGPA ...
9,dataset_0,dataset,10 + Blockchain Nodes / Masternodes to set up ...,10,20,13.718507,Banking,"B.S., Finance – Arizona State University, W.P...."
